# Тестирование API эндпоинтов

Этот notebook содержит тесты для всех эндпоинтов API из `model/main.py`

## Эндпоинты для тестирования:
1. `GET /` - приветственный эндпоинт
2. `POST /start_talk/` - начало диалога с LLM
3. `POST /get_user_info/` - получение информации о пользователе
4. `POST /get_profession_info/` - получение информации о профессии через RAG


In [4]:
import requests
import json
from datetime import datetime
from typing import Dict, Any

# Базовый URL API
BASE_URL = "http://localhost:8000"

# Тестовые данные
TEST_USER_ID = "test_user_123"
TEST_PROMPT = "Привет, я хочу узнать о профессиях в IT"
TEST_PROFESSION_NAME = "Программист"


## 1. Тест GET / - Приветственный эндпоинт


In [5]:
def test_root_endpoint():
    """Тест корневого эндпоинта"""
    print("=" * 50)
    print("Тест: GET /")
    print("=" * 50)
    
    try:
        response = requests.get(f"{BASE_URL}/")
        print(f"Статус код: {response.status_code}")
        print(f"Ответ: {json.dumps(response.json(), ensure_ascii=False, indent=2)}")
        
        assert response.status_code == 200, f"Ожидался статус 200, получен {response.status_code}"
        assert "message" in response.json(), "В ответе отсутствует поле 'message'"
        
        print("✅ Тест пройден успешно!")
        return True
    except requests.exceptions.ConnectionError:
        print("❌ Ошибка: Не удалось подключиться к API. Убедитесь, что сервер запущен на http://localhost:8000")
        return False
    except Exception as e:
        print(f"❌ Ошибка: {e}")
        return False

test_root_endpoint()


Тест: GET /
Статус код: 200
Ответ: {
  "message": "Welcome to the Model API"
}
✅ Тест пройден успешно!


True

## 2. Тест POST /start_talk/ - Начало диалога с LLM


In [6]:
def test_start_talk_endpoint(user_id: str = TEST_USER_ID, prompt: str = TEST_PROMPT):
    """Тест эндпоинта начала диалога"""
    print("=" * 50)
    print("Тест: POST /start_talk/")
    print("=" * 50)
    
    payload = {
        "user_id": user_id,
        "prompt": prompt,
        "parameters": {}
    }
    
    try:
        print(f"Отправка запроса с user_id={user_id}, prompt='{prompt[:50]}...'")
        response = requests.post(
            f"{BASE_URL}/start_talk/",
            json=payload,
            headers={"Content-Type": "application/json"}
        )
        
        print(f"Статус код: {response.status_code}")
        
        if response.status_code == 200:
            result = response.json()
            print(f"Ответ LLM: {result.get('msg', '')[:200]}...")
            print(f"Профессии в ответе: {result.get('professions') is not None}")
            if result.get('professions'):
                print(f"Количество профессий: {len(result.get('professions', {}))}")
                print(f"Список профессий: {list(result.get('professions', {}).keys())}")
            
            assert "msg" in result, "В ответе отсутствует поле 'msg'"
            print("✅ Тест пройден успешно!")
            return result
        else:
            print(f"❌ Ошибка: Статус код {response.status_code}")
            print(f"Ответ сервера: {response.text}")
            return None
            
    except requests.exceptions.ConnectionError:
        print("❌ Ошибка: Не удалось подключиться к API. Убедитесь, что сервер запущен на http://localhost:8000")
        return None
    except Exception as e:
        print(f"❌ Ошибка: {e}")
        import traceback
        traceback.print_exc()
        return None

# Запуск теста
result = test_start_talk_endpoint()


Тест: POST /start_talk/
Отправка запроса с user_id=test_user_123, prompt='Привет, я хочу узнать о профессиях в IT...'
Статус код: 200
Ответ LLM: Ошибка: <AioRpcError of RPC that terminated with:
	code = StatusCode.UNAUTHENTICATED
	details = "Unknown api key 'b1ga****jipk (D2A9AE84)'"
	debug_error_string = "UNKNOWN:Error received from peer ipv4...
Профессии в ответе: False
✅ Тест пройден успешно!


## 3. Тест POST /get_user_info/ - Получение информации о пользователе


In [5]:
def test_get_user_info_endpoint(user_id: str = TEST_USER_ID):
    """Тест эндпоинта получения информации о пользователе"""
    print("=" * 50)
    print("Тест: POST /get_user_info/")
    print("=" * 50)
    
    payload = {
        "user_id": user_id,
        "prompt": "",  # Для этого эндпоинта prompt не обязателен
        "parameters": {}
    }
    
    try:
        print(f"Отправка запроса с user_id={user_id}")
        response = requests.post(
            f"{BASE_URL}/get_user_info/",
            json=payload,
            headers={"Content-Type": "application/json"}
        )
        
        print(f"Статус код: {response.status_code}")
        
        if response.status_code == 200:
            result = response.json()
            print(f"Информация о пользователе:")
            print(json.dumps(result, ensure_ascii=False, indent=2))
            
            assert "user_id" in result, "В ответе отсутствует поле 'user_id'"
            assert "msg" in result, "В ответе отсутствует поле 'msg'"
            assert "timestamp" in result, "В ответе отсутствует поле 'timestamp'"
            
            print("✅ Тест пройден успешно!")
            return result
        else:
            print(f"❌ Ошибка: Статус код {response.status_code}")
            print(f"Ответ сервера: {response.text}")
            return None
            
    except requests.exceptions.ConnectionError:
        print("❌ Ошибка: Не удалось подключиться к API. Убедитесь, что сервер запущен на http://localhost:8000")
        return None
    except Exception as e:
        print(f"❌ Ошибка: {e}")
        import traceback
        traceback.print_exc()
        return None

# Запуск теста
user_info = test_get_user_info_endpoint()


Тест: POST /get_user_info/
Отправка запроса с user_id=test_user_123
Статус код: 200
Информация о пользователе:
{
  "user_id": "test_user_123",
  "msg": "{'test_user_123': {'user_state': 'who', 'user_type': None, 'user_metadata': None}}",
  "timestamp": "2025-11-22 16:21:02.187782"
}
✅ Тест пройден успешно!


## 4. Тест POST /get_profession_info/ - Получение информации о профессии через RAG


In [6]:
def test_get_profession_info_endpoint(user_id: str = TEST_USER_ID, profession_name: str = TEST_PROFESSION_NAME):
    """Тест эндпоинта получения информации о профессии"""
    print("=" * 50)
    print("Тест: POST /get_profession_info/")
    print("=" * 50)
    
    payload = {
        "user_id": user_id,
        "profession_name": profession_name
    }
    
    try:
        print(f"Отправка запроса с user_id={user_id}, profession_name='{profession_name}'")
        response = requests.post(
            f"{BASE_URL}/get_profession_info/",
            json=payload,
            headers={"Content-Type": "application/json"}
        )
        
        print(f"Статус код: {response.status_code}")
        
        if response.status_code == 200:
            result = response.json()
            print(f"Ответ RAG системы: {result.get('msg', '')[:300]}...")
            print(f"Профессии в ответе: {result.get('professions') is not None}")
            if result.get('professions'):
                print(f"Количество профессий: {len(result.get('professions', {}))}")
            
            assert "msg" in result, "В ответе отсутствует поле 'msg'"
            print("✅ Тест пройден успешно!")
            return result
        else:
            print(f"❌ Ошибка: Статус код {response.status_code}")
            print(f"Ответ сервера: {response.text}")
            return None
            
    except requests.exceptions.ConnectionError:
        print("❌ Ошибка: Не удалось подключиться к API. Убедитесь, что сервер запущен на http://localhost:8000")
        return None
    except Exception as e:
        print(f"❌ Ошибка: {e}")
        import traceback
        traceback.print_exc()
        return None

# Запуск теста
profession_info = test_get_profession_info_endpoint()


Тест: POST /get_profession_info/
Отправка запроса с user_id=test_user_123, profession_name='Программист'
Статус код: 500
❌ Ошибка: Статус код 500
Ответ сервера: Internal Server Error


## 5. Комплексное тестирование - Последовательность запросов

Тестируем полный сценарий использования API


In [8]:
def run_comprehensive_test():
    """Запуск всех тестов последовательно"""
    print("=" * 70)
    print("КОМПЛЕКСНОЕ ТЕСТИРОВАНИЕ API")
    print("=" * 70)
    
    test_user_id = f"test_user_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    results = {}
    
    # Тест 1: Корневой эндпоинт
    print("\n[1/4] Тестирование GET /")
    results['root'] = test_root_endpoint()
    
    # Тест 2: Начало диалога
    print("\n[2/4] Тестирование POST /start_talk/")
    results['start_talk'] = test_start_talk_endpoint(
        user_id=test_user_id,
        prompt="Я интересуюсь программированием и хочу узнать о карьере в IT"
    )
    
    # Тест 3: Информация о пользователе
    print("\n[3/4] Тестирование POST /get_user_info/")
    results['user_info'] = test_get_user_info_endpoint(user_id=test_user_id)
    
    # Тест 4: Информация о профессии
    print("\n[4/4] Тестирование POST /get_profession_info/")
    results['profession_info'] = test_get_profession_info_endpoint(
        user_id=test_user_id,
        profession_name="Data Scientist"
    )
    
    # Итоговая статистика
    print("\n" + "=" * 70)
    print("ИТОГИ ТЕСТИРОВАНИЯ")
    print("=" * 70)
    passed = sum(1 for r in results.values() if r is not None and r is not False)
    total = len(results)
    print(f"Пройдено тестов: {passed}/{total}")
    
    for test_name, result in results.items():
        status = "✅ PASSED" if (result is not None and result is not False) else "❌ FAILED"
        print(f"  {test_name}: {status}")
    
    return results

# Раскомментируйте следующую строку для запуска комплексного теста
comprehensive_results = run_comprehensive_test()


КОМПЛЕКСНОЕ ТЕСТИРОВАНИЕ API

[1/4] Тестирование GET /
Тест: GET /
Статус код: 200
Ответ: {
  "message": "Welcome to the Model API"
}
✅ Тест пройден успешно!

[2/4] Тестирование POST /start_talk/
Тест: POST /start_talk/
Отправка запроса с user_id=test_user_20251122_162540, prompt='Я интересуюсь программированием и хочу узнать о ка...'
Статус код: 200
Ответ LLM: Здравствуйте! Меня зовут консультант по профориентации. Чтобы лучше понять, как я могу вам помочь, позвольте задать несколько вопросов.

1. Подскажите, пожалуйста, ваше имя....
Профессии в ответе: False
✅ Тест пройден успешно!

[3/4] Тестирование POST /get_user_info/
Тест: POST /get_user_info/
Отправка запроса с user_id=test_user_20251122_162540
Статус код: 200
Информация о пользователе:
{
  "user_id": "test_user_20251122_162540",
  "msg": "{'test_user_20251122_162540': {'user_state': 'who', 'user_type': None, 'user_metadata': None}}",
  "timestamp": "2025-11-22 16:25:47.574128"
}
✅ Тест пройден успешно!

[4/4] Тестирование POST

## 6. Дополнительные тесты с различными параметрами

Можно протестировать API с разными входными данными


In [9]:
# Тест с разными промптами
test_prompts = [
    "Расскажи о профессии программиста",
    "Какие навыки нужны для работы в data science?",
    "Хочу стать веб-разработчиком, с чего начать?",
]

print("Тестирование /start_talk/ с разными промптами:")
print("=" * 70)

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n[Тест {i}/{len(test_prompts)}]")
    result = test_start_talk_endpoint(
        user_id=f"test_user_prompt_{i}",
        prompt=prompt
    )
    print("-" * 70)


Тестирование /start_talk/ с разными промптами:

[Тест 1/3]
Тест: POST /start_talk/
Отправка запроса с user_id=test_user_prompt_1, prompt='Расскажи о профессии программиста...'
Статус код: 200
Ответ LLM: Здравствуйте! Меня зовут консультант по профориентации. Чтобы лучше понять, как ответить на ваш вопрос, позвольте задать несколько уточняющих вопросов.

1. Подскажите, пожалуйста, как вас зовут?...
Профессии в ответе: False
✅ Тест пройден успешно!
----------------------------------------------------------------------

[Тест 2/3]
Тест: POST /start_talk/
Отправка запроса с user_id=test_user_prompt_2, prompt='Какие навыки нужны для работы в data science?...'
Статус код: 200
Ответ LLM: Здравствуйте! Прежде чем ответить на ваш вопрос, позвольте задать несколько уточняющих вопросов. Как вас зовут?...
Профессии в ответе: False
✅ Тест пройден успешно!
----------------------------------------------------------------------

[Тест 3/3]
Тест: POST /start_talk/
Отправка запроса с user_id=test_user_p

## 7. Проверка документации API

FastAPI автоматически генерирует документацию, доступную по адресу:
- Swagger UI: http://localhost:8000/docs
- ReDoc: http://localhost:8000/redoc


In [10]:
def check_api_docs():
    """Проверка доступности документации API"""
    print("Проверка доступности документации API...")
    
    endpoints_to_check = [
        ("/docs", "Swagger UI"),
        ("/redoc", "ReDoc"),
        ("/openapi.json", "OpenAPI Schema")
    ]
    
    for endpoint, name in endpoints_to_check:
        try:
            response = requests.get(f"{BASE_URL}{endpoint}")
            if response.status_code == 200:
                print(f"✅ {name} доступен: {BASE_URL}{endpoint}")
            else:
                print(f"⚠️ {name} вернул статус {response.status_code}")
        except Exception as e:
            print(f"❌ {name} недоступен: {e}")

# Раскомментируйте для проверки документации
check_api_docs()


Проверка доступности документации API...
✅ Swagger UI доступен: http://localhost:8000/docs
✅ ReDoc доступен: http://localhost:8000/redoc
✅ OpenAPI Schema доступен: http://localhost:8000/openapi.json
